___

# <font color= #99C8F5> **Predicción del Tipo de Cambio en Banxico** </font>
#### <font color= #2E9AFE> `Modelos no Lineales para Pronósticos`</font>
<Strong> Daniela De la Torre, Samantha Sánchez, Sofía Maldonado & Viviana Toledo </Strong>

_04/03/2026._

___

# <font color= #99C8F5> **Librerías y Cargado de Datos** </font>

In [ ]:
# Generales
import pandas as pd
import banxicoapi

# Visualización
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Preprocesamiento
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Modelo
from statsmodels.tsa.statespace.sarimax import SARIMAX


In [ ]:
# Conexión a la API del Banco de México

# Guardar el token de acceso obtenido del sitio de Banxico
token = "f9f264f2cf8962d6273f74db64e602e1ec126aac885d9f11ddcc24e3166a5743"  # ¡IMPORTANTE!: Reemplaza 'TU_TOKEN_AQUI' con tu token real de Banxico
# Crear objeto API con el token (usando la librería banxicoapi)
api = banxicoapi.BanxicoApi(token)
# Identificador de la serie del tipo de cambio FIX (peso/dólar)
series = ["SF43718"]  # Banxico: SF43718 corresponde al tipo de cambio FIX.
# Consultar todos los datos históricos de la serie
respuesta = api.get(series)  # Devuelve un JSON con los datos de la serie
# Convertir a DataFrame de pandas
datos = pd.DataFrame(respuesta[0]['datos'])  # Extraemos la lista de dicts con 'fecha' y 'dato'
# Renombrar columnas y convertir tipos
datos.rename(columns={'fecha':'Fecha','dato':'Valor'}, inplace=True)
datos['Fecha'] = pd.to_datetime(datos['Fecha'], format='%d/%m/%Y')   # Fecha a formato datetime
datos['Valor'] = pd.to_numeric(datos['Valor'], errors='coerce')       # Tipo de dato numérico (float)
datos.set_index('Fecha', inplace=True)  # Definir la fecha como índice

In [6]:
datos

,Valor
Fecha,
1991-11-12,3.0735
1991-11-13,3.0712
1991-11-14,3.0718
1991-11-15,3.0684
1991-11-18,3.0673
...,...
2026-02-24,17.1910
2026-02-25,17.1700
2026-02-26,17.2563


# <font color= #99C8F5> **Preprocesamiento** </font>

In [7]:
# Re-muestrear la serie a frecuencia diaria y rellenar vacíos
datos = datos.resample('D').ffill()  # Rellenar fines de semana con el último valor conocido
datos.dropna(inplace=True)           # Eliminar filas vacías si las hubiera al inicio

In [ ]:
# Prueba de Dickey-Fuller aumentado
resultado_adf = adfuller(datos['Valor'])
print(f"ADF Statistic: {resultado_adf[0]}, p-value: {resultado_adf[1]}")
# Si p-value > 0.05 indicaría que la serie no es estacionaria (necesita diferenciación)

# Diferenciar la serie una vez (d=1) si es necesario
datos_diff = datos['Valor'].diff().dropna()

# Opcional: Graficar ACF y PACF para determinar p y q (visual, no se muestra aquí)
#plot_acf(datos_diff, lags=30)
#plot_pacf(datos_diff, lags=30)


ADF Statistic: -1.658920931433397, p-value: 0.45239214951818885


In [ ]:

# Definir y ajustar el modelo SARIMAX con parámetros elegidos
# (p,d,q)=(1,1,1) y (P,D,Q,s)=(1,1,1,5) en este ejemplo
modelo = SARIMAX(datos['Valor'], order=(1,1,1), seasonal_order=(1,1,1,5))
resultado = modelo.fit(disp=False)
print(resultado.summary())  # Resumen del ajuste (AIC, coeficientes, etc.)


                                     SARIMAX Results                                     
Dep. Variable:                             Valor   No. Observations:                12530
Model:             SARIMAX(1, 1, 1)x(1, 1, 1, 5)   Log Likelihood               12617.337
Date:                           Tue, 03 Mar 2026   AIC                         -25224.674
Time:                                   00:42:55   BIC                         -25187.497
Sample:                               11-12-1991   HQIC                        -25212.229
                                    - 03-02-2026                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.4699      0.169      2.775      0.006       0.138       0.802
ma.L1         -0.4548      0.170     -2.672

In [10]:
# Predecir los próximos 9 días (5 al 13 de marzo de 2026)
pronostico = resultado.get_forecast(steps=9)
pred_mean = pronostico.predicted_mean

# Crear índice de fechas para el pronóstico (a partir del último día observado)
fechas_pred = pd.date_range(start=datos.index[-1] + pd.Timedelta(days=1), periods=9, freq='D')

# Crear DataFrame con Fecha y Pronóstico
df_pred = pd.DataFrame({'Fecha': fechas_pred, 'Pronóstico': pred_mean.values})

# Exportar las predicciones a un archivo Excel
df_pred.to_excel("predicciones_mar5_13_2026.xlsx", index=False)


In [ ]:

fig = go.Figure()
# Serie histórica observada
fig.add_trace(go.Scatter(x=datos.index, y=datos['Valor'], mode='lines', name='Observado'))
# Pronóstico futuro
fig.add_trace(go.Scatter(x=df_pred['Fecha'], y=df_pred['Pronóstico'], mode='lines', name='Pronóstico'))
fig.update_layout(title='Tipo de cambio MXN/USD (FIX): Observado vs Pronóstico',
                  xaxis_title='Fecha', yaxis_title='Tipo de cambio (FIX)')
fig.show()
